In [3]:
import os, sys
import pandas as pd 
import numpy as np

from dotenv import load_dotenv

import requests 
import json 

from bs4 import BeautifulSoup
import io

import csv

import time


In [3]:
libs = [pd, requests]

for lib in libs:
    print(f"{lib.__name__}=={lib.__version__}")

pandas==3.0.0
requests==2.33.0


In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent   # notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))
RAW_DIR = PROJECT_ROOT / "data" / "raw"

In [6]:
# CHMI data
# 10 min data
# get_chmi_weather_data()

In [7]:
## CHMI hourly data: using different function rn
# get_chmi_weather_data_hourly()
# fails for Klementinum, Brdy

In [9]:
# utility 
def print_json_structure(obj, indent=0):
    pad = "  " * indent
    if isinstance(obj, dict):
        for key, value in obj.items():
            print(f"{pad}{key}: {type(value).__name__}")
            print_json_structure(value, indent + 1)
    elif isinstance(obj, list):
        print(f"{pad}[list] len={len(obj)}")
        if obj:
            print_json_structure(obj[0], indent + 1)

def print_json_structure_from_url(url):
    obj = requests.get(url).json()
    print_json_structure(obj)


In [10]:
#data chmi
print_json_structure_from_url("https://opendata.chmi.cz/meteorology/climate/historical/metadata/meta1.json")

zaznamID: str
datovyZdrojID: str
datovyTokID: str
datumVytvoreni: str
verzeDat: str
data: dict
  type: str
  data: dict
    header: str
    values: list
      [list] len=5539
        [list] len=8


In [11]:
def get_chmi_weather_stations_metadata(out_path: Path|str = RAW_DIR / "chmi_weather_stations_metadata.csv"):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://opendata.chmi.cz/"
    route = "/meteorology/climate/historical/metadata/meta1.json"
    headers = {
        "accept": "application/json",
        "User-Agent": "JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)",
    }
    resp = requests.get(f"{url}{route}", headers=headers, timeout=60)
    resp.raise_for_status()
    response = resp.json()
    data_response = response.get('data', {}).get('data', {})
    headers = data_response.get('header', '').split(',')
    values = data_response.get('values', [])
    df = pd.DataFrame(values, columns=headers)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    return df


chmi_weather_stations = get_chmi_weather_stations_metadata()


In [12]:
chmi_weather_stations.head()

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION
0,0-20000-0-04030,ZIS04030,2015-01-01T00:00:00Z,3999-12-31T23:59:00Z,Reykjavik,-21.903905,64.127653,51.0
1,0-20000-0-11406,L3CHEB01,1863-10-01T00:00:00Z,1919-12-31T23:59:00Z,Cheb,12.362892,50.076212,458.0
2,0-20000-0-11406,L3CHEB01,1933-05-07T00:00:00Z,1938-04-30T23:59:00Z,Cheb,12.388900,50.0739,471.0
3,0-20000-0-11406,L3CHEB01,1943-06-01T00:00:00Z,1945-01-31T23:59:00Z,Cheb,12.388900,50.0739,471.0
4,0-20000-0-11406,L3CHEB01,1951-01-01T00:00:00Z,1960-12-31T23:59:00Z,Cheb,12.388900,50.0739,471.0


In [11]:
### chmi data variables
print_json_structure_from_url("https://opendata.chmi.cz/meteorology/climate/historical/metadata/meta2.json")

zaznamID: str
datovyZdrojID: str
datovyTokID: str
datumVytvoreni: str
verzeDat: str
data: dict
  type: str
  data: dict
    header: str
    values: list
      [list] len=80798
        [list] len=9


In [12]:
def get_chmi_weather_variables_metadata(out_path: Path|str = RAW_DIR / "chmi_weather_variables_metadata.csv"):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://opendata.chmi.cz/"
    route = "/meteorology/climate/historical/metadata/meta2.json"
    headers = {
        "accept": "application/json",
        "User-Agent": "JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)",
    }
    resp = requests.get(f"{url}{route}", headers=headers, timeout=60)
    resp.raise_for_status()
    response = resp.json()
    data_response = response.get('data', {}).get('data', {})
    headers = data_response.get('header', '').split(',')
    values = data_response.get('values', [])
    df = pd.DataFrame(values, columns=headers)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    return df

df_chmi_vars = get_chmi_weather_variables_metadata()

In [13]:
df_chmi_vars.head()

,OBS_TYPE,WSI,BEGIN_DATE,END_DATE,EG_EL_ABBREVIATION,NAME,UN_DESCRIPTION,HEIGHT,SCHEDULE
0,DLY,0-20000-0-11406,1863-10-01T00:00:00Z,1919-12-31T23:59:00Z,T,Teplota,°C,2.0,"AVG,06:00,13:00,20:00"
1,DLY,0-20000-0-11406,1865-06-01T00:00:00Z,1919-12-31T23:59:00Z,TMI,Teplota min,°C,2.0,20:00
2,DLY,0-20000-0-11406,1865-06-01T00:00:00Z,1919-12-31T23:59:00Z,TMA,Teplota max,°C,2.0,20:00
3,DLY,0-20000-0-11406,1933-05-07T00:00:00Z,1938-04-30T23:59:00Z,SSV,Sluneční svit,hod,2.0,00:00
4,DLY,0-20000-0-11406,1943-06-01T00:00:00Z,1945-01-31T23:59:00Z,SSV,Sluneční svit,hod,2.0,00:00


In [ ]:
### chmi weather data

# 10 min data download

In [14]:
print_json_structure_from_url("https://opendata.chmi.cz/meteorology/climate/historical/data/10min/2025/10m-0-203-0-11514-202511.json")

zaznamID: str
datovyZdrojID: str
datovyTokID: str
datumVytvoreni: str
verzeDat: str
data: dict
  type: str
  data: dict
    header: str
    values: list
      [list] len=60474
        [list] len=6


In [ ]:
### this download all data (for 10min periods)

def get_chmi_weather_data(start_year=2025, end_year=2025, wsi_csv = RAW_DIR / "wsi_dict.csv", out_path: Path|str = RAW_DIR / "weather_data_10min.csv"):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    base_url = "https://opendata.chmi.cz/"
    route_template = "/meteorology/climate/historical/data/10min/{year}/10m-{wsi}-{ym}.json"
    url_header = {
        "accept": "application/json",
        "User-Agent": "JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)",
    }
    # we set to dowload only data from selected stations to not get unnecessary big dataset
    wsi_dict = pd.read_csv(wsi_csv, encoding="utf-8-sig").set_index("key")["value"].to_dict()
    years = [f"{y}" for y in range(start_year, end_year + 1)]
    months = [f"{m:02d}" for m in range(1, 13)]
    results = []
    for wsi in wsi_dict:
        for year in years:
            for month in months:
                ym = f"{year}{month}"
                route = route_template.format(year=year, wsi=wsi, ym=ym)
                try:
                    time.sleep(0.5)
                    response = requests.get(f"{base_url}{route}", headers=url_header, timeout=90)
                    response.raise_for_status()
                except requests.exceptions.RequestException as exc:
                    print(f"Request failed for {wsi} {wsi_dict.get(wsi)} {year} {month}: {exc}")
                    continue
                response = response.json()
                data_response = response.get("data", {}).get("data", {})
                headers = data_response.get("header", "").split(",")
                values = data_response.get("values", [])
                if not values:
                    continue
                df_part = pd.DataFrame(values, columns=headers)
                df_part["WSI"] = wsi
                df_part["YEAR"] = year
                df_part["MONTH"] = month
                results.append(df_part)
    if not results:
        return pd.DataFrame()
    df = pd.concat(results, ignore_index=True)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    return df
    
df_weather_10min = get_chmi_weather_data()

In [15]:
df_weather_10min.head()

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH
0,0-203-0-10904013001,H,2025-01-01T00:00:00Z,93.0,,0.0,0-203-0-10904013001,2025,01
1,0-203-0-10904013001,H,2025-01-01T00:10:00Z,93.0,,0.0,0-203-0-10904013001,2025,01
2,0-203-0-10904013001,H,2025-01-01T00:20:00Z,93.0,,0.0,0-203-0-10904013001,2025,01
3,0-203-0-10904013001,H,2025-01-01T00:30:00Z,93.0,,0.0,0-203-0-10904013001,2025,01
4,0-203-0-10904013001,H,2025-01-01T00:40:00Z,93.0,,0.0,0-203-0-10904013001,2025,01


In [19]:
df_weather_10min.shape

(4992840, 9)

In [ ]:
### !!! update dict 
# first dowload metadata, update dict in data porcessing ntb and then proceed to weather data download

In [21]:
### weather data for 1hour -- if needed

## - good, but if i understand it correctly, it take values only at whole hour, which might be probelmatic
#  e.g. for Tmax or SRA, which should be rather averaged, summed etc.

def get_chmi_weather_data_hourly_from_10_min(
    start_year=2025,
    end_year=2025,
    wsi_csv=RAW_DIR / "wsi_dict.csv",
    out_path: Path | str = RAW_DIR / "weather_data_10min_hourly.csv"
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    base_url = "https://opendata.chmi.cz/"
    route_template = "/meteorology/climate/historical/data/10min/{year}/10m-{wsi}-{ym}.json"

    url_header = {
        "accept": "application/json",
        "User-Agent": "JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)",
    }

    # we set to download only data from selected stations to not get unnecessary big dataset
    wsi_dict = pd.read_csv(wsi_csv, encoding="utf-8-sig").set_index("key")["value"].to_dict()

    years = [f"{y}" for y in range(start_year, end_year + 1)]
    months = [f"{m:02d}" for m in range(1, 13)]

    results = []

    for wsi in wsi_dict:
        for year in years:
            for month in months:
                month = f"{int(month):02d}"
                ym = f"{year}{month}"

                route = route_template.format(year=year, wsi=wsi, ym=ym)

                try:
                    time.sleep(0.5)
                    response = requests.get(f"{base_url}{route}", headers=url_header, timeout=90)
                    response.raise_for_status()

                except requests.exceptions.RequestException as exc:
                    print(f"Request failed for {wsi} {wsi_dict.get(wsi)} {year} {month}: {exc}")
                    continue

                response = response.json()

                data_response = response.get("data", {}).get("data", {})
                headers = data_response.get("header", "").split(",")
                values = data_response.get("values", [])

                if not values:
                    continue

                df_part = pd.DataFrame(values, columns=headers)

                # Convert timestamp and keep only full-hour observations
                df_part["DT"] = pd.to_datetime(df_part["DT"], utc=True)

                df_part = df_part[
                    (df_part["DT"].dt.minute == 0) &
                    (df_part["DT"].dt.second == 0)
                ].copy()

                df_part["WSI"] = wsi
                df_part["YEAR"] = year
                df_part["MONTH"] = month

                results.append(df_part)

    if not results:
        return pd.DataFrame()

    df = pd.concat(results, ignore_index=True)

    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    return df
df_weather = get_chmi_weather_data_hourly_from_10_min()


In [22]:
df_weather.shape

(832140, 9)

In [17]:
station_counts = (
    df_weather["STATION"]
    .value_counts()
    .reset_index()
)

station_counts.columns = ["STATION", "n_observations"]

station_counts

,STATION,n_observations
0,0-20000-0-11520,183948
1,0-20000-0-11567,175188
2,0-20000-0-11519,131388
3,0-20000-0-11518,131388
4,0-203-0-11514,122628
5,0-203-0-10904013001,52560
6,0-203-0-11101007001,26280
7,0-203-0-11201024001,8760
8,0-203-0-11515,8760


In [ ]:
### Download the hourly data
# It has different variables (from ceilometer) and doesnt have temperature etc, I would not use it


def get_chmi_weather_data_hourly(
    start_year=2025,
    end_year=2025,
    wsi_csv=RAW_DIR / "wsi_dict.csv",
    out_path: Path | str = RAW_DIR / "weather_data_1hour.csv"
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    base_url = "https://opendata.chmi.cz/"
    route_template = "/meteorology/climate/historical/data/1hour/{year}/1h-{wsi}-{ym}.json"
    url_header = {
        "accept": "application/json",
        "User-Agent": "JEM207 DataProcessingCourse (Educational access)",
    }
    wsi_dict = (
        pd.read_csv(wsi_csv, encoding="utf-8-sig")
        .set_index("key")["value"]
        .to_dict()
    )
    years = [f"{y}" for y in range(start_year, end_year + 1)]
    months =  ["11"] #[f"{m:02d}" for m in range(1, 13)]
    results = []
    for wsi in wsi_dict:
        for year in years:
            for month in months:
                ym = f"{year}{month}"
                route = route_template.format(year=year, wsi=wsi, ym=ym)
                try:
                    time.sleep(0.5)
                    response = requests.get(
                        f"{base_url}{route}",
                        headers=url_header,
                        timeout=90
                    )
                    response.raise_for_status()
                except requests.exceptions.RequestException as exc:
                    print(
                        f"Request failed for {wsi} "
                        f"{wsi_dict.get(wsi)} {year}-{month}: {exc}"
                    )
                    continue
                response_json = response.json()
                data_response = response_json.get("data", {}).get("data", {})
                headers = data_response.get("header", "").split(",")
                values = data_response.get("values", [])
                if not values:
                    print(f"No data for {wsi} {wsi_dict.get(wsi)} {year}-{month}")
                    continue
                df_part = pd.DataFrame(values, columns=headers)
                df_part["WSI"] = wsi
                df_part["YEAR"] = year
                df_part["MONTH"] = month
                results.append(df_part)
                print(
                    f"Loaded {wsi} {wsi_dict.get(wsi)} "
                    f"{year}-{month}: {len(df_part)} rows"
                )
    if not results:
        return pd.DataFrame()
    df = pd.concat(results, ignore_index=True)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    return df
df_weather = get_chmi_weather_data_hourly()


In [7]:
def download_airquality_metadata(metadata_url="https://opendata.chmi.cz/air_quality/recent/metadata/metadata.json",
                                  out_path: Path|str = RAW_DIR / "airquality_CHMI_stations_metadata.csv"):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    resp = requests.get(metadata_url, timeout=60)
    resp.raise_for_status()
    metadata = resp.json()
    mapping_list = []
    localities = metadata.get("data", {}).get("Localities", [])
    for locality in localities:
        loc_code = locality.get("LocalityCode", {})
        loc_name = locality.get("Name", {})
        loc = locality.get("Localization", {})
        lon = loc.get("LonAsNumber")
        lat = loc.get("LatAsNumber")
        alt = loc.get("Alt")
        addr = locality.get("Address", {})
        street = addr.get("Street")
        city = addr.get("City")
        programs = locality.get("MeasuringPrograms", [])
        for program in programs:
            station_code = program.get("Code")
            measurements = program.get("Measurements", [])
            for measurement in measurements:
                row = {
                    "id_registration": measurement.get("IdRegistration"),
                    "station_code": station_code,
                    "locality_code": loc_code,
                    "locality_name": loc_name,
                    "street": street,
                    "city": city,
                    "lon": lon,
                    "lat": lat,
                    "alt": alt,
                    "component_code": measurement.get("ComponentCode"),
                    "component_name": measurement.get("ComponentName"),
                    "unit": measurement.get("UnitAsASCII"),
                }
                mapping_list.append(row)
    df_mapping = pd.DataFrame(mapping_list)
    df_mapping.to_csv(out_path, index=False, encoding="utf-8-sig")
    return df_mapping

air_qual_meta = download_airquality_metadata()

In [8]:
air_qual_meta.head()

,id_registration,station_code,locality_code,locality_name,street,city,lon,lat,alt,component_code,component_name,unit
0,40555,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,SO2,oxid siřičitý,ug/m^3
1,40557,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,NO2,oxid dusičitý,ug/m^3
2,40560,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,NOx,oxidy dusíku,ug/m^3
3,40559,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,O3,přízemní ozon,ug/m^3
4,40561,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,PM10,částice PM10,ug/m^3


In [9]:
air_qual_meta = air_qual_meta[air_qual_meta['locality_name'].str.contains('Praha', case=False, na=False)]

In [10]:
# air quality data 2025
def download_airquality_data_period(
    start_year=2025,
    end_year=2025,
    data_dir_url="https://opendata.chmi.cz/air_quality/recent/data/",
    out_path: Path|str = RAW_DIR / "airquality_CHMI_1hour.csv",
    ids_to_keep=None
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    resp = requests.get(data_dir_url, timeout=60)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    csv_files = [
        link.get("href")
        for link in soup.find_all("a")
        if link.get("href", "").endswith(".csv")
    ]
    if not csv_files:
        raise FileNotFoundError("No CSV files found in the directory.")
    
    years = [f"{y}" for y in range(start_year, end_year + 1)]
    months = [f"{m:02d}" for m in range(1, 13)]

    file_prefixes = [
        f"airquality_1h_avg_CZ_{year}{month}"
        for year in years
        for month in months
    ]
    files_period = [
        file for file in csv_files
        if any(file.startswith(prefix) for prefix in file_prefixes)
    ]
    files_period = sorted(files_period)
    if not files_period:
        raise FileNotFoundError(
            f"No air-quality CSV files found for years {start_year}-{end_year} and months {months}."
        )
    print(f"Found {len(files_period)} hourly air-quality files.")
    results = []
    for i, file in enumerate(files_period, start=1):
        file_url = f"{data_dir_url}{file}"
        try:
            time.sleep(0.5) 
            data_response = requests.get(file_url, timeout=90) 
            data_response.raise_for_status()
        except requests.exceptions.RequestException as exc:
            print(f"Request failed for {file}: {exc}")
            continue
        df_part = pd.read_csv(io.StringIO(data_response.text))
        if ids_to_keep is not None:
            df_part = df_part[df_part["idRegistration"].isin(ids_to_keep)].copy()
        if df_part.empty:
            continue
        df_part["source_file"] = file
        results.append(df_part)
    if not results:
        return pd.DataFrame()
    df_data = pd.concat(results, ignore_index=True)
    df_data.to_csv(out_path, index=False, encoding="utf-8-sig")
    return df_data

In [11]:
air_stat_dict = dict(zip(air_qual_meta["id_registration"], air_qual_meta["locality_name"]))

air_stat_df = (
    pd.DataFrame(air_stat_dict.items(), columns=["id_registration", "locality_name"])
    .sort_values("id_registration")
)

air_stat_df.to_csv(RAW_DIR / "air_stat_dict.csv", index=False, encoding="utf-8-sig")

In [ ]:
# !!! beware, it runs for about 40 minutes for the whole dataset

In [ ]:
ids_to_keep = list(air_stat_dict)

df_air_qual = download_airquality_data_period(ids_to_keep=ids_to_keep)


Found 8753 hourly air-quality files.
Request failed for airquality_1h_avg_CZ_2025011219.csv: HTTPSConnectionPool(host='opendata.chmi.cz', port=443): Max retries exceeded with url: /air_quality/recent/data/airquality_1h_avg_CZ_2025011219.csv (Caused by ConnectTimeoutError(<HTTPSConnection(host='opendata.chmi.cz', port=443) at 0x1e031f120d0>, 'Connection to opendata.chmi.cz timed out. (connect timeout=90)'))
Request failed for airquality_1h_avg_CZ_2025012313.csv: HTTPSConnectionPool(host='opendata.chmi.cz', port=443): Max retries exceeded with url: /air_quality/recent/data/airquality_1h_avg_CZ_2025012313.csv (Caused by ConnectTimeoutError(<HTTPSConnection(host='opendata.chmi.cz', port=443) at 0x1e031f116d0>, 'Connection to opendata.chmi.cz timed out. (connect timeout=90)'))
Request failed for airquality_1h_avg_CZ_2025012605.csv: HTTPSConnectionPool(host='opendata.chmi.cz', port=443): Max retries exceeded with url: /air_quality/recent/data/airquality_1h_avg_CZ_2025012605.csv (Caused by Co

In [20]:
df_air_qual

,idRegistration,startTime,idValueType,value,source_file
0,10221,2025-11-01T00:00:00Z,8,1.3,airquality_1h_avg_CZ_2025110100.csv
1,40237,2025-11-01T00:00:00Z,8,48.7,airquality_1h_avg_CZ_2025110100.csv
2,40238,2025-11-01T00:00:00Z,8,6.2,airquality_1h_avg_CZ_2025110100.csv
3,40244,2025-11-01T00:00:00Z,8,12.6,airquality_1h_avg_CZ_2025110100.csv
4,40257,2025-11-01T00:00:00Z,6,-5009.0,airquality_1h_avg_CZ_2025110100.csv
...,...,...,...,...,...
345294,2053564,2025-11-30T23:00:00Z,6,-5009.0,airquality_1h_avg_CZ_2025113023.csv
345295,2140455,2025-11-30T23:00:00Z,6,-5009.0,airquality_1h_avg_CZ_2025113023.csv
345296,2140459,2025-11-30T23:00:00Z,6,-5009.0,airquality_1h_avg_CZ_2025113023.csv
345297,2140463,2025-11-30T23:00:00Z,6,-5009.0,airquality_1h_avg_CZ_2025113023.csv


In [23]:
#### chmi air quality
# only one file

def download_airquality_data_snapshot(data_dir_url="https://opendata.chmi.cz/air_quality/recent/data/", 
                             out_path: Path|str = RAW_DIR / "airquality_CHMI_data_snapshot.csv"):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    resp = requests.get(data_dir_url, timeout=60)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    csv_files = [link.get("href") for link in soup.find_all("a") if link.get("href", "").endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("No CSV files found in the directory.")
    latest_file = sorted(csv_files)[-1]
    file_url = f"{data_dir_url}{latest_file}"
    print(f"Downloading raw data from: {latest_file}")
    data_response = requests.get(file_url)
    data_response.raise_for_status()
    df_data = pd.read_csv(io.StringIO(data_response.text))
    df_data.to_csv(out_path, index=False, encoding="utf-8-sig")
    return df_data

df_air_qual_snapshot = download_airquality_data_snapshot()


In [14]:
df_air_qual_snapshot.head()

,idRegistration,startTime,idValueType,value
0,10221,2026-05-25T05:00:00Z,8,5.9
1,40237,2026-05-25T05:00:00Z,8,25.1
2,40238,2026-05-25T05:00:00Z,8,70.0
3,40244,2026-05-25T05:00:00Z,8,79.0
4,40257,2026-05-25T05:00:00Z,8,4.0
